In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
from typing import Dict, List, Tuple
import sys

In [2]:
sys.path.insert(0, '..')

In [7]:
from src.REPTree import (
    REPTreeClassifier,
    REPTreeRegressor,
    DataPreprocessor,
    REPTreePipeline
)
from src.REPTree.utils import train_test_split, export_text, plot_tree_stats
from src.REPTree.metrics import (
    accuracy_score,
    confusion_matrix,
    mean_squared_error,
    r2_score,
    mean_absolute_error
)

In [8]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

In [9]:
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [10]:
ROOT = Path.cwd().parent     
DATA_DIR = ROOT / "data"

dataset_path = DATA_DIR / "classification/iris_extended.csv"

df = pd.read_csv(dataset_path)
df.head(1000)

,species,elevation,soil_type,sepal_length,sepal_width,petal_length,petal_width,sepal_area,petal_area,sepal_aspect_ratio,...,sepal_to_petal_length_ratio,sepal_to_petal_width_ratio,sepal_petal_length_diff,sepal_petal_width_diff,petal_curvature_mm,petal_texture_trichomes_per_mm2,leaf_area_cm2,sepal_area_sqrt,petal_area_sqrt,area_ratios
0,setosa,161.8,sandy,5.16,3.41,1.64,0.26,17.5956,0.4264,1.513196,...,3.146341,13.115385,3.52,3.15,5.33,18.33,53.21,4.194711,0.652993,41.265478
1,setosa,291.4,clay,5.48,4.05,1.53,0.37,22.1940,0.5661,1.353086,...,3.581699,10.945946,3.95,3.68,5.90,20.45,52.53,4.711051,0.752396,39.205087
2,setosa,144.3,sandy,5.10,2.80,1.47,0.38,14.2800,0.5586,1.821429,...,3.469388,7.368421,3.63,2.42,5.66,24.62,50.25,3.778889,0.747395,25.563910
3,setosa,114.6,clay,4.64,3.44,1.53,0.17,15.9616,0.2601,1.348837,...,3.032680,20.235294,3.11,3.27,4.51,22.91,50.85,3.995197,0.510000,61.367166
4,setosa,110.9,loamy,4.85,2.87,1.23,0.26,13.9195,0.3198,1.689895,...,3.943089,11.038462,3.62,2.61,4.03,21.56,40.57,3.730885,0.565509,43.525641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,virginica,95.7,clay,9.12,2.93,6.37,2.63,26.7216,16.7531,3.112628,...,1.431711,1.114068,2.75,0.30,9.75,11.88,76.03,5.169294,4.093055,1.595024
996,virginica,193.5,clay,6.77,3.14,5.60,1.67,21.2578,9.3520,2.156051,...,1.208929,1.880240,1.17,1.47,13.40,11.51,66.45,4.610618,3.058104,2.273075
997,virginica,209.4,clay,5.35,3.11,6.19,1.84,16.6385,11.3896,1.720257,...,0.864297,1.690217,-0.84,1.27,10.10,9.73,60.61,4.079032,3.374848,1.460850
998,virginica,189.2,sandy,5.11,2.56,4.30,1.86,13.0816,7.9980,1.996094,...,1.188372,1.376344,0.81,0.70,12.21,8.96,58.23,3.616849,2.828074,1.635609


In [11]:
X = df.drop(columns=["species"]).dropna(how='all')
y = df["species"].dropna(how='all')

In [12]:
y.values

array(['setosa', 'setosa', 'setosa', ..., 'virginica', 'virginica',
       'virginica'], shape=(1200,), dtype=object)

In [13]:
class LabelEncoder:
    """
    Simple label encoder implemented from scratch.
    
    Maps categorical labels to integer classes (0, 1, 2, ...).
    """

    def __init__(self):
        self.classes_ = None
        self.class_to_index_ = None

    def fit(self, y):
        """
        Learn the unique classes and create a mapping.
        
        Parameters
        ----------
        y : list or array-like
            List of labels (strings, ints, etc.)
        """
        # Convert to list so it works with numpy or pandas
        y = list(y)

        # Unique classes sorted alphabetically (just like sklearn)
        self.classes_ = sorted(set(y))

        # Mapping: label -> integer index
        self.class_to_index_ = {label: idx for idx, label in enumerate(self.classes_)}

        return self

    def transform(self, y):
        """
        Transform labels into integer encoded values.
        
        Parameters
        ----------
        y : list or array-like
        
        Returns
        -------
        list of int
        """
        y = list(y)
        encoded = []

        for label in y:
            if label not in self.class_to_index_:
                raise ValueError(f"Unknown label '{label}'. Fit encoder first or check your data.")
            encoded.append(self.class_to_index_[label])

        return encoded

    def fit_transform(self, y):
        """
        Fit the encoder and return transformed labels.
        """
        self.fit(y)
        return self.transform(y)

    def inverse_transform(self, y):
        """
        Convert integer labels back to original labels.
        
        Parameters
        ----------
        y : list or array-like of ints
        
        Returns
        -------
        list of original labels
        """
        index_to_class = {idx: label for idx, label in enumerate(self.classes_)}

        decoded = []
        for idx in y:
            if idx not in index_to_class:
                raise ValueError(f"Unknown class index '{idx}'.")
            decoded.append(index_to_class[idx])

        return decoded


In [14]:
X = X.values
y = y.values

In [15]:
categorical_cols = [i for i in range(X.shape[1]) 
                    if not np.issubdtype(type(X[0, i]), np.number)]
categorical_cols

[1]

In [16]:
y_encoder = LabelEncoder()
y = y_encoder.fit_transform(y)
y = np.array(y)

In [17]:
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[:, col] = le.fit_transform(X[:, col])
    encoders[col] = le 
X = X.astype(float)

In [18]:
X,y

(array([[161.8       ,   2.        ,   5.16      , ...,   4.19471096,
           0.65299311,  41.26547842],
        [291.4       ,   0.        ,   5.48      , ...,   4.71105084,
           0.75239617,  39.20508744],
        [144.3       ,   2.        ,   5.1       , ...,   3.77888873,
           0.74739548,  25.56390977],
        ...,
        [ 73.6       ,   0.        ,   6.79      , ...,   4.69760577,
           3.26606797,   2.06872469],
        [239.6       ,   2.        ,   6.38      , ...,   3.78037035,
           3.01048169,   1.576873  ],
        [201.5       ,   1.        ,   5.16      , ...,   4.06349603,
           2.83992958,   2.04731439]], shape=(1200, 20)),
 array([0, 0, 0, ..., 2, 2, 2], shape=(1200,)))

In [19]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [20]:
y_test,X_test

(array([0, 1, 1, 0, 1, 0, 2, 1, 0, 2, 2, 2, 0, 2, 0, 0, 0, 1, 1, 2, 1, 1,
        1, 0, 2, 1, 1, 2, 2, 2, 0, 0, 2, 1, 0, 0, 0, 0, 2, 2, 0, 2, 1, 2,
        1, 1, 0, 0, 1, 0, 1, 2, 1, 1, 2, 1, 2, 0, 0, 2, 0, 0, 2, 1, 2, 1,
        1, 1, 2, 1, 0, 2, 1, 1, 2, 1, 0, 1, 2, 1, 0, 0, 0, 2, 1, 0, 1, 2,
        0, 1, 0, 2, 1, 1, 0, 1, 0, 0, 0, 2, 1, 0, 2, 2, 2, 0, 1, 1, 2, 0,
        0, 1, 2, 0, 1, 2, 0, 1, 0, 0, 1, 0, 0, 2, 2, 1, 1, 0, 1, 2, 2, 1,
        2, 0, 2, 0, 2, 2, 0, 0, 1, 1, 2, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 2,
        0, 2, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 2, 0, 1, 1, 0, 1, 2, 2, 0,
        0, 0, 2, 1, 1, 0, 1, 2, 0, 0, 1, 2, 0, 2, 0, 2, 1, 0, 0, 1, 0, 1,
        0, 1, 2, 1, 1, 1, 1, 0, 2, 0, 1, 0, 0, 0, 2, 2, 2, 0, 1, 0, 2, 2,
        2, 0, 0, 1, 2, 2, 0, 0, 0, 2, 2, 0, 2, 2, 1, 1, 2, 1, 1, 1]),
 array([[ 54.5       ,   2.        ,   5.        , ...,   3.80788655,
           0.58172158,  42.84869976],
        [288.8       ,   0.        ,   5.8       , ...,   4.0514195 ,
           2

In [21]:
print(f"\nDataset sizes:")
print(f"  Training: {len(X_train)} samples")
print(f"  Validation: {len(X_val)} samples")
print(f"  Test: {len(X_test)} samples")
print(f"  Features: {X.shape[1]}")
print(f"  Classes: {len(np.unique(y))}")


Dataset sizes:
  Training: 720 samples
  Validation: 240 samples
  Test: 240 samples
  Features: 20
  Classes: 3


In [22]:
clf_no_prune = REPTreeClassifier(
        criterion='gini',
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42
    )
clf_no_prune.fit(X_train, y_train)

In [23]:
train_acc_no_prune = clf_no_prune.score(X_train, y_train)
val_acc_no_prune = clf_no_prune.score(X_val, y_val)
test_acc_no_prune = clf_no_prune.score(X_test, y_test)

In [24]:
print("\n   Without Pruning Results:")
print(f"   Train accuracy: {train_acc_no_prune:.4f}")
print(f"   Validation accuracy: {val_acc_no_prune:.4f}")
print(f"   Test accuracy: {test_acc_no_prune:.4f}")
print(f"   Tree depth: {clf_no_prune.get_depth()}")
print(f"   Number of nodes: {clf_no_prune.tree_.count_nodes()}")
print(f"   Number of leaves: {clf_no_prune.get_n_leaves()}")


   Without Pruning Results:
   Train accuracy: 1.0000
   Validation accuracy: 0.9708
   Test accuracy: 0.9708
   Tree depth: 8
   Number of nodes: 41
   Number of leaves: 21


In [25]:
clf_with_prune = REPTreeClassifier(
        criterion='gini',
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        pruning='rep',
        random_state=42
    )
clf_with_prune.fit(X_train, y_train, X_val=X_val, y_val=y_val)

In [26]:
train_acc_prune = clf_with_prune.score(X_train, y_train)
val_acc_prune = clf_with_prune.score(X_val, y_val)
test_acc_prune = clf_with_prune.score(X_test, y_test)

In [27]:
print("\n   With REP Pruning Results:")
print(f"   Train accuracy: {train_acc_prune:.4f}")
print(f"   Validation accuracy: {val_acc_prune:.4f}")
print(f"   Test accuracy: {test_acc_prune:.4f}")
print(f"   Tree depth: {clf_with_prune.get_depth()}")
print(f"   Number of nodes: {clf_with_prune.tree_.count_nodes()}")
print(f"   Number of leaves: {clf_with_prune.get_n_leaves()}")


   With REP Pruning Results:
   Train accuracy: 0.9819
   Validation accuracy: 0.9792
   Test accuracy: 0.9792
   Tree depth: 4
   Number of nodes: 11
   Number of leaves: 6


# Regression

In [28]:
dataset_path = DATA_DIR / "regression/day.csv"
df = pd.read_csv(dataset_path)
df

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
726,727,2012-12-27,1,1,12,0,4,1,2,0.254167,0.226642,0.652917,0.350133,247,1867,2114
727,728,2012-12-28,1,1,12,0,5,1,2,0.253333,0.255046,0.590000,0.155471,644,2451,3095
728,729,2012-12-29,1,1,12,0,6,0,2,0.253333,0.242400,0.752917,0.124383,159,1182,1341
729,730,2012-12-30,1,1,12,0,0,0,1,0.255833,0.231700,0.483333,0.350754,364,1432,1796


In [29]:
X = df.drop(columns=["cnt"]).dropna(how='all')
y = df["cnt"].dropna(how='all')

In [30]:
X = X.values
y = y.values

In [31]:
categorical_cols = [i for i in range(X.shape[1]) 
                    if not np.issubdtype(type(X[0, i]), np.number)]
categorical_cols

[1]

In [32]:
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[:, col] = le.fit_transform(X[:, col])
    encoders[col] = le 
X = X.astype(float)

In [33]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [34]:
y_test.shape,X_test.shape

((146,), (146, 15))

In [35]:
print(f"\nDataset sizes:")
print(f"  Training: {len(X_train)} samples")
print(f"  Validation: {len(X_val)} samples")
print(f"  Test: {len(X_test)} samples")
print(f"  Features: {X.shape[1]}")
print(f"  Classes: {len(np.unique(y))}")


Dataset sizes:
  Training: 439 samples
  Validation: 146 samples
  Test: 146 samples
  Features: 15
  Classes: 696


In [36]:
reg_no_prune = REPTreeRegressor(
        criterion='variance',
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    )
reg_no_prune.fit(X_train, y_train)

In [37]:
y_train_pred_no_prune = reg_no_prune.predict(X_train)
y_val_pred_no_prune = reg_no_prune.predict(X_val)
y_test_pred_no_prune = reg_no_prune.predict(X_test)
    
train_mse_no_prune = mean_squared_error(y_train, y_train_pred_no_prune)
val_mse_no_prune = mean_squared_error(y_val, y_val_pred_no_prune)
test_mse_no_prune = mean_squared_error(y_test, y_test_pred_no_prune)
    
train_r2_no_prune = r2_score(y_train, y_train_pred_no_prune)
val_r2_no_prune = r2_score(y_val, y_val_pred_no_prune)
test_r2_no_prune = r2_score(y_test, y_test_pred_no_prune)

In [38]:
print("\n   Without Pruning Results:")
print(f"   Train MSE: {train_mse_no_prune:.4f}, R²: {train_r2_no_prune:.4f}")
print(f"   Validation MSE: {val_mse_no_prune:.4f}, R²: {val_r2_no_prune:.4f}")
print(f"   Test MSE: {test_mse_no_prune:.4f}, R²: {test_r2_no_prune:.4f}")
print(f"   Tree depth: {reg_no_prune.get_depth()}")
print(f"   Number of nodes: {reg_no_prune.tree_.count_nodes()}")
print(f"   Number of leaves: {reg_no_prune.get_n_leaves()}")


   Without Pruning Results:
   Train MSE: 4532.6281, R²: 0.9988
   Validation MSE: 62201.4001, R²: 0.9832
   Test MSE: 43422.0402, R²: 0.9891
   Tree depth: 12
   Number of nodes: 309
   Number of leaves: 155


In [39]:
reg_with_prune = REPTreeRegressor(
        criterion='variance',
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        pruning='rep',
        random_state=42
    )
reg_with_prune.fit(X_train, y_train, X_val=X_val, y_val=y_val)

In [40]:
y_train_pred_prune = reg_with_prune.predict(X_train)
y_val_pred_prune = reg_with_prune.predict(X_val)
y_test_pred_prune = reg_with_prune.predict(X_test)
    
train_mse_prune = mean_squared_error(y_train, y_train_pred_prune)
val_mse_prune = mean_squared_error(y_val, y_val_pred_prune)
test_mse_prune = mean_squared_error(y_test, y_test_pred_prune)
    
train_r2_prune = r2_score(y_train, y_train_pred_prune)
val_r2_prune = r2_score(y_val, y_val_pred_prune)
test_r2_prune = r2_score(y_test, y_test_pred_prune)

In [41]:
print("\n   With REP Pruning Results:")
print(f"   Train MSE: {train_mse_prune:.4f}, R²: {train_r2_prune:.4f}")
print(f"   Validation MSE: {val_mse_prune:.4f}, R²: {val_r2_prune:.4f}")
print(f"   Test MSE: {test_mse_prune:.4f}, R²: {test_r2_prune:.4f}")
print(f"   Tree depth: {reg_with_prune.get_depth()}")
print(f"   Number of nodes: {reg_with_prune.tree_.count_nodes()}")
print(f"   Number of leaves: {reg_with_prune.get_n_leaves()}")


   With REP Pruning Results:
   Train MSE: 13353.2593, R²: 0.9964
   Validation MSE: 44196.2102, R²: 0.9881
   Test MSE: 40937.8807, R²: 0.9897
   Tree depth: 11
   Number of nodes: 229
   Number of leaves: 115
